[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/12-observability.ipynb)

## Colab setup

On Google Colab, use the setup in `tutorial/common/colab.py`. A fresh Colab runtime does not have this repository, so clone the repo into the session first. Then, before the lesson cells, run `setup_colab` from that module (import it from `tutorial.common.colab` and call it, or open the file and run `setup_colab` from there). The helper installs the tutorial packages, checks the repo out under `/content/book-agents` when it is still missing, and copies a Colab secret named `OPENAI_API_KEY`. Local Jupyter and VS Code can skip it when you already have the repo.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) before you run `setup_colab`. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

# 12. Observability

Each span names three identities: the customer, the Harbor Jar seller, and the tool. The chili-oil question from tutorial 2 runs again so the same work has a trace. Nothing secret belongs in the JSON.

The next sections define the three functions. `span_status` turns a tool result into a status word. `make_span` records one tool. `root_span` records the turn. `run_turn` calls the last two through `HOOKS`.

## Setup

Find the repo so `import tutorial` works, then load the key. `chat` calls the OpenAI Chat Completions API. The key is not printed.

In [ ]:
import sys
from pathlib import Path


def find_root():
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "tutorial" / "common" / "client.py").is_file():
            return folder
    raise RuntimeError("Open this notebook inside the book-agents repository.")


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.runtime import require_key

_, model = require_key()
print("model:", model)

## Status of one tool result

`span_status`, defined in the next cell, reads one tool result and returns a short status word. `make_span` stores that word. The result text stays out of the span, so a secret in the tool output is not a field this function returns.

1. **Enter only for three prefixes.** The function looks at the start of the result. The prefixes are `ERROR`, `DENIED`, and `CONFIRM_REQUIRED`. A result that starts with none of them returns `ok`. A catalog line, a shipping sentence, and an empty string are `ok`. The letters `DENIED` later in an ordinary sentence stay `ok`, because the later checks run only after one of those three prefixes matched.

2. **A confirm is waiting.** If the result starts with `CONFIRM_REQUIRED`, the status is `confirm_required`. The gate is waiting for a person. This return happens before the denial check, so a confirm stays a confirm.

3. **A denial is a denial.** If the result starts with `DENIED`, the status is `denied`. If it started with `ERROR` and the letters `DENIED` appear in the first 80 characters, the status is also `denied`. Those 80 characters cover a short reason on the first line.

4. **What remains is an error.** A result that started with `ERROR`, and did not match the denial check, returns `error`.

5. **Return that one word.** The function returns `confirm_required`, `denied`, `error`, or `ok`. It does not return the tool text.

In [ ]:
def span_status(result):
    if result.startswith("ERROR") or result.startswith("DENIED") or result.startswith("CONFIRM_REQUIRED"):
        if result.startswith("CONFIRM_REQUIRED"):
            return "confirm_required"
        if result.startswith("DENIED") or "DENIED" in result[:80]:
            return "denied"
        return "error"
    return "ok"

## One tool span

`make_span`, defined in the next cell, builds one span for one tool call. `run_turn` calls it after the tool returns, with the trace id, the step, the tool name, and the result. The dict names the customer, the Harbor Jar seller, and the tool. The result is passed only to `span_status`. Secret-shaped values in the result are not copied into the JSON.

1. **Keep the trace id.** The caller passes `trace_id`. Every span in this turn shares it.

2. **Build the span id.** It is the trace id, the step number, and the tool name, joined by hyphens. A catalog lookup on step 1 of `trace-chili-ohio` is `trace-chili-ohio-1-query_catalog`.

3. **Name the service and the span.** `service` is `harbor-jar`. `name` is `tool.` plus the tool name, so that lookup is `tool.query_catalog`.

4. **Set the status.** `span_status(result)` supplies `ok`, `error`, `denied`, or `confirm_required`. The result string is not a key in the dict.

5. **Record the step.** `step` is the model-call number the loop was on when the tool ran.

6. **Name three identities.** `user` is kind `user` and id `customer`. `agent` is kind `agent` and id `harbor-jar-concierge`. `actor` is kind `tool` and id the tool name, because this span is that tool's work.

7. **Register the hook.** The last line is `set_hook("make_span", make_span)`. `run_turn` looks up `"make_span"` and appends this dict after each tool result. Until this cell has run, the loop records no tool spans. This function does not import the loop, and the loop does not import this notebook.

In [ ]:
from tutorial.common.hooks import set_hook

def make_span(trace_id, step, name, result):
    """One tool span. Identity is explicit. Secret-shaped values are not copied in."""
    return {
        "trace_id": trace_id,
        "span_id": trace_id + "-" + str(step) + "-" + name,
        "service": "harbor-jar",
        "name": "tool." + name,
        "status": span_status(result),
        "step": step,
        "user": {"kind": "user", "id": "customer"},
        "agent": {"kind": "agent", "id": "harbor-jar-concierge"},
        "actor": {"kind": "tool", "id": name},
    }


# run_turn in loop.py calls this name. Registering here turns tool spans on.
set_hook("make_span", make_span)

## The turn span

`root_span`, defined in the next cell, builds the span for the whole turn. `run_turn` appends it once, before the first model call. No tool has returned yet, so the status is `ok`. The actor is the seller.

1. **Keep the trace id.** The tool spans from `make_span` use this same id.

2. **Build the span id.** It is the trace id plus `-turn`. For `trace-chili-ohio` the id is `trace-chili-ohio-turn`.

3. **Name the service and the span.** `service` is `harbor-jar`. `name` is `concierge.turn`.

4. **Status is ok.** This span is the turn. A tool verdict belongs on the tool span.

5. **Name three identities.** The user is the customer. The agent is `harbor-jar-concierge`. The actor is that same agent, because the turn is the seller's work. A tool span is the place where the actor is a tool.

6. **Register the hook.** The last line is `set_hook("root_span", root_span)`. `run_turn` looks up `"root_span"` and appends this span first. `system_text` sees that name on `HOOKS` and tells the model the work is traced as the customer using the Harbor Jar seller, and to leave secrets out of the answer. Tutorial 13 and tutorial 15 define these same functions and register both hooks, so those lessons can run alone and still record who acted.

In [ ]:
from tutorial.common.hooks import set_hook

def root_span(trace_id):
    return {
        "trace_id": trace_id,
        "span_id": trace_id + "-turn",
        "service": "harbor-jar",
        "name": "concierge.turn",
        "status": "ok",
        "user": {"kind": "user", "id": "customer"},
        "agent": {"kind": "agent", "id": "harbor-jar-concierge"},
        "actor": {"kind": "agent", "id": "harbor-jar-concierge"},
    }


# run_turn in loop.py calls this name. Registering here turns the turn span on.
set_hook("root_span", root_span)

## Chili oil, with a trace id

`span_status`, `make_span`, and `root_span` are defined above, and the two hooks are registered. This cell imports the catalog and the topic lookup, the tools from the earlier lessons. The Ohio question runs again with trace id `trace-chili-ohio`. You read the spans. Every span names the customer and the seller. At least one span has a tool actor. The answer is still grounded in the catalog and the shipping topic. The JSON does not carry a secret.

In [ ]:
import json

import tutorial.common.facts
import tutorial.common.catalog

from tutorial.common.get_db import reset_db
from tutorial.common.harness import check, system_text
from tutorial.common.loop import run_agent

reset_db()
result = run_agent(
    "Do you have Calabrian chili oil, and can you ship a jar to Ohio?",
    system_text(),
    trace_id="trace-chili-ohio",
)
print("ANSWER:", result["text"])
print(json.dumps(result["spans"], indent=2, sort_keys=True))
blob = json.dumps(result["spans"])
check(result["stopped"] == "final", "the traced question finished")
check(len(result["spans"]) >= 2, "the turn span and at least one tool span were recorded")
check(all(span["user"]["id"] == "customer" for span in result["spans"]), "every span names the customer")
check(all(span["agent"]["id"] == "harbor-jar-concierge" for span in result["spans"]), "every span names the seller")
check(any(span["actor"]["kind"] == "tool" for span in result["spans"]), "a tool has its own actor")
check("sk-" not in blob and "canary" not in blob.lower(), "the trace does not carry a secret")